In [31]:
# ============================================================
# CELL 1: Load only first 50k polygons from parks.tsv
# (unchanged from v1)
# ============================================================

import numpy as np
from shapely import wkt as shapely_wkt
from collections import Counter

DATA_PATH = "/raid/ruban/data/parks.tsv"
POLY_COUNT = 50000

def _parse_tsv_line(line: str):
    parts = line.rstrip("\n").split("\t")
    if len(parts) < 2:
        raise ValueError("bad tsv line")
    feat_id = int(parts[0])
    wkt_str = parts[1]
    return feat_id, wkt_str

def _linestring_to_polygon(geom):
    from shapely.geometry import Polygon
    coords = list(geom.coords)
    if len(coords) == 0:
        raise ValueError("empty geometry")
    if len(coords) == 2:
        raise ValueError("LineString too short for polygon conversion: 2 coords")
    if coords[0] != coords[-1]:
        coords.append(coords[0])
    if len(coords) < 4:
        raise ValueError(f"closed ring too short: {len(coords)} coords")
    return Polygon(coords)

def _geom_to_polygon(geom):
    if geom.is_empty:
        raise ValueError("empty geometry")
    gtype = geom.geom_type
    if gtype == "Polygon":
        return geom
    if gtype == "MultiPolygon":
        parts = list(geom.geoms)
        if not parts:
            raise ValueError("empty multipolygon")
        return max(parts, key=lambda g: g.area)
    if gtype == "LineString":
        return _linestring_to_polygon(geom)
    raise ValueError(f"unsupported geometry type: {gtype}")

geometries = []
poly_ids = []
skip_reasons = Counter()

with open(DATA_PATH, "r", encoding="utf-8") as fh:
    for lineno, line in enumerate(fh):
        if len(geometries) >= POLY_COUNT:
            break
        try:
            feat_id, wkt_str = _parse_tsv_line(line)
            geom = shapely_wkt.loads(wkt_str)
            geom = _geom_to_polygon(geom)
            if not geom.is_valid:
                geom = geom.buffer(0)
            if geom.is_empty:
                skip_reasons["empty_after_fix"] += 1
                continue
            geometries.append(geom)
            poly_ids.append(feat_id)
        except Exception as exc:
            msg = str(exc).lower()
            if "too short" in msg:
                skip_reasons["degenerate_linestring"] += 1
            elif "empty" in msg:
                skip_reasons["empty_geometry"] += 1
            else:
                skip_reasons["other"] += 1

print("Loaded polygons:", len(geometries))
print("Skipped:", sum(skip_reasons.values()))
print("Sample geom type:", geometries[0].geom_type if geometries else None)
print("Sample poly id:", poly_ids[0] if poly_ids else None)

Loaded polygons: 50000
Skipped: 52
Sample geom type: Polygon
Sample poly id: 4061698


In [32]:
# ============================================================
# CELL 2: Normalize rings + build shape descriptors in parallel
# GT descriptor: 256-pt resampling + PCA axis alignment
# (rotation-invariant, 512-d)
# ============================================================

import numpy as np
import multiprocessing as mp

NUM_WORKERS  = min(100, mp.cpu_count())
RESAMPLE_PTS = 256  # 256 pts -> 512-d descriptor

def _remove_duplicate_last(coords):
    coords = np.asarray(coords, dtype=np.float32)
    if len(coords) >= 2 and np.allclose(coords[0], coords[-1]):
        coords = coords[:-1]
    return coords

def _normalize_ring(coords):
    coords = _remove_duplicate_last(coords)
    if len(coords) < 3:
        return None
    coords = coords - coords.mean(axis=0, keepdims=True)
    r = np.linalg.norm(coords, axis=1).max()
    if r <= 1e-12:
        return None
    return (coords / r).astype(np.float32)

def _resample_ring(coords, n_points):
    coords = _remove_duplicate_last(coords)
    if coords is None or len(coords) < 3:
        return None
    closed  = np.vstack([coords, coords[0]])
    segs    = closed[1:] - closed[:-1]
    lens    = np.linalg.norm(segs, axis=1)
    total   = lens.sum()
    if total <= 1e-12:
        return None
    cum     = np.concatenate([[0.0], np.cumsum(lens)])
    targets = np.linspace(0.0, total, n_points, endpoint=False)
    out, j  = [], 0
    for t in targets:
        while j + 1 < len(cum) and cum[j + 1] < t:
            j += 1
        seg_len = lens[j]
        p = closed[j].copy() if seg_len <= 1e-12 else \
            closed[j] + ((t - cum[j]) / seg_len) * (closed[j + 1] - closed[j])
        out.append(p)
    return np.asarray(out, dtype=np.float32)

def _canonicalize_start(points):
    idx = np.lexsort((points[:, 1], points[:, 0]))[0]
    return np.roll(points, -idx, axis=0)

def _descriptor_from_geom(geom):
    try:
        coords = np.asarray(geom.exterior.coords, dtype=np.float32)
        coords = _normalize_ring(coords)
        if coords is None:
            return None

        # rotate to PCA frame -> rotation-invariant canonicalization
        cov            = np.cov(coords.T)
        _, eigvecs     = np.linalg.eigh(cov)
        coords         = coords @ eigvecs[:, ::-1]  # major axis -> x

        if coords[:, 0].mean() < 0:
            coords[:, 0] *= -1
        if coords[:, 1].mean() < 0:
            coords[:, 1] *= -1

        pts = _resample_ring(coords, RESAMPLE_PTS)
        if pts is None:
            return None
        pts  = _canonicalize_start(pts)
        desc = pts.reshape(-1).astype(np.float32)
        nrm  = np.linalg.norm(desc)
        if nrm <= 1e-12:
            return None
        return desc / nrm
    except Exception:
        return None

print(f"Building descriptors with {NUM_WORKERS} workers...")

with mp.Pool(processes=NUM_WORKERS) as pool:
    descriptors = pool.map(_descriptor_from_geom, geometries, chunksize=64)

valid_mask       = [d is not None for d in descriptors]
valid_indices    = [i for i, ok in enumerate(valid_mask) if ok]
descriptors      = np.vstack([descriptors[i] for i in valid_indices]).astype(np.float32)
geometries_valid = [geometries[i] for i in valid_indices]
poly_ids_valid   = [poly_ids[i]   for i in valid_indices]

print("Valid descriptors:", descriptors.shape[0])
print("Descriptor dim   :", descriptors.shape[1])
print("Filtered out     :", POLY_COUNT - len(valid_indices))
print("Sample poly id   :", poly_ids_valid[0])

Building descriptors with 100 workers...


Valid descriptors: 49963
Descriptor dim   : 512
Filtered out     : 37
Sample poly id   : 4061698


In [33]:
# CELL 3: Build cosine-based shape GT (descriptor cosine, no IoU)

import numpy as np

DATA_END_50K    = 40000
QUERY_START_50K = 40000
QUERY_END_50K   = 50000
NUM_GT_QUERIES  = 1000
TOPK_GT         = 500

valid_indices_arr = np.asarray(valid_indices, dtype=np.int32)

db_rows        = np.where(valid_indices_arr < DATA_END_50K)[0]
query_rows_all = np.where((valid_indices_arr >= QUERY_START_50K) &
                          (valid_indices_arr < QUERY_END_50K))[0]
query_rows     = query_rows_all[:NUM_GT_QUERIES]

X_db = descriptors[db_rows]
X_q  = descriptors[query_rows]

db_global_ids    = valid_indices_arr[db_rows]
query_global_ids = valid_indices_arr[query_rows]

print("DB descriptors    :", X_db.shape)
print("Query descriptors :", X_q.shape)

sim     = X_q @ X_db.T
topk_idx = np.argpartition(-sim, kth=TOPK_GT-1, axis=1)[:, :TOPK_GT]
row_ids  = np.arange(topk_idx.shape[0])[:, None]
order    = np.argsort(-sim[row_ids, topk_idx], axis=1)
topk_sorted = topk_idx[row_ids, order]

custom_gt = {}
for r, q_gid in enumerate(query_global_ids):
    custom_gt[int(q_gid)] = db_global_ids[topk_sorted[r]].astype(int).tolist()

print("GT queries:", len(custom_gt))
print("Sample neighbors:", custom_gt[int(query_global_ids[0])][:10])

DB descriptors    : (39970, 512)
Query descriptors : (1000, 512)
GT queries: 1000
Sample neighbors: [6121, 1739, 15503, 7931, 39286, 37264, 30913, 21252, 22582, 16268]


In [34]:
sample_q = next(iter(custom_gt))
print("GT neighbors stored for sample query:", len(custom_gt[sample_q]))

GT neighbors stored for sample query: 500


In [35]:
# ============================================================
# CELL 4 [FIX 2]: Build PolyMP graphs with rotation-invariant
# node features.
#
# WHY: v1 used raw (x,y) coordinates and absolute edge angle
# arctan2(dy,dx). Two congruent polygons rotated relative to
# each other get entirely different node features -> the model
# cannot learn rotation-invariant similarity.
#
# NEW FEATURES per vertex i (6-d, all rotation-invariant):
#   0: distance from centroid (scale-normalized)
#   1: outgoing edge length
#   2: incoming edge length
#   3: turning angle at vertex (signed, radians) -- invariant
#   4: sin(turning angle)  -- smooth encoding
#   5: cos(turning angle)  -- smooth encoding
# ============================================================

import numpy as np
import torch
from torch_geometric.data import Data

def normalize_polygon(coords):
    coords = coords - coords.mean(axis=0, keepdims=True)
    scale = np.linalg.norm(coords, axis=1).max()
    if scale > 0:
        coords = coords / scale
    return coords.astype(np.float32)

def build_features_and_edges_from_geom(geom):
    coords = np.asarray(geom.exterior.coords, dtype=np.float32)
    # drop closing duplicate vertex
    if len(coords) >= 2 and np.allclose(coords[0], coords[-1]):
        coords = coords[:-1]
    coords = normalize_polygon(coords)
    N = len(coords)

    # centroid distances (already centred by normalize_polygon)
    dists = np.linalg.norm(coords, axis=1)  # [N]

    # edge vectors
    next_coords = np.roll(coords, -1, axis=0)  # coords[(i+1)%N]
    prev_coords = np.roll(coords,  1, axis=0)  # coords[(i-1)%N]

    out_edges = next_coords - coords            # outgoing edge at i
    in_edges  = coords - prev_coords            # incoming edge at i

    out_lens = np.linalg.norm(out_edges, axis=1).clip(1e-12)  # [N]
    in_lens  = np.linalg.norm(in_edges,  axis=1).clip(1e-12)  # [N]

    # turning angle: signed angle from incoming to outgoing direction
    # cross product gives sin, dot product gives cos -> atan2 gives signed angle
    in_unit  = in_edges  / in_lens[:, None]
    out_unit = out_edges / out_lens[:, None]

    cross = in_unit[:, 0] * out_unit[:, 1] - in_unit[:, 1] * out_unit[:, 0]  # sin
    dot   = (in_unit * out_unit).sum(axis=1)                                   # cos
    turning = np.arctan2(cross, dot)  # [N], rotation-invariant

    # stack 6 rotation-invariant features
    x = np.stack([
        dists,
        out_lens,
        in_lens,
        turning,
        np.sin(turning),
        np.cos(turning),
    ], axis=1).astype(np.float32)  # [N, 6]

    # bidirectional ring edges
    idx = np.arange(N)
    src = np.concatenate([idx, (idx + 1) % N])
    dst = np.concatenate([(idx + 1) % N, idx])
    edge_index = np.stack([src, dst], axis=0).astype(np.int64)

    return x, edge_index

graphs = []
graph_global_ids = []

for geom, gidx in zip(geometries_valid, valid_indices):
    x, edge_index = build_features_and_edges_from_geom(geom)
    graphs.append(
        Data(
            x=torch.tensor(x, dtype=torch.float32),
            edge_index=torch.tensor(edge_index, dtype=torch.long),
            poly_id=int(gidx)
        )
    )
    graph_global_ids.append(int(gidx))

global_to_graphrow = {gid: i for i, gid in enumerate(graph_global_ids)}

db_graph_rows    = [global_to_graphrow[int(gid)] for gid in db_global_ids
                    if int(gid) in global_to_graphrow]
query_graph_rows = [global_to_graphrow[int(gid)] for gid in query_global_ids
                    if int(gid) in global_to_graphrow]

db_graph_global_ids    = [graph_global_ids[i] for i in db_graph_rows]
query_graph_global_ids = [graph_global_ids[i] for i in query_graph_rows]

print("DB graph rows   :", len(db_graph_rows))
print("Query graph rows:", len(query_graph_rows))
print("Graphs built    :", len(graphs))
print("Sample graph    :", graphs[0])
print("Sample global id:", graph_global_ids[0])

DB graph rows   : 39970
Query graph rows: 1000
Graphs built    : 49963
Sample graph    : Data(x=[56, 6], edge_index=[2, 112], poly_id=0)
Sample global id: 0


In [36]:
# ============================================================
# CELL 5 [FIX 3]: Improved PolyMP model
#
# CHANGES vs v1:
#   - hidden_dim 64 -> 128  (more capacity)
#   - 3 -> 4 GNN layers     (deeper message passing)
#   - readout: global_mean_pool + global_max_pool concatenated
#     (max catches salient vertices, mean gives global summary)
#   - BatchNorm after each conv (stabilises training)
#   - Residual connection where dims match
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_mean, scatter_max
from torch_geometric.nn import global_mean_pool, global_max_pool

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class PolyMPConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.mlp_msg = nn.Sequential(
            nn.Linear(in_dim * 2, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.mlp_update = nn.Sequential(
            nn.Linear(in_dim + out_dim, out_dim),
            nn.ReLU(),
        )
        self.bn = nn.BatchNorm1d(out_dim)
        # residual projection only when dims differ
        self.res = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else nn.Identity()

    def forward(self, x, edge_index):
        row, col = edge_index
        m = torch.cat([x[row], x[col]], dim=1)
        m = self.mlp_msg(m)
        agg = scatter_mean(m, col, dim=0, dim_size=x.size(0))
        out = self.mlp_update(torch.cat([x, agg], dim=1))
        out = self.bn(out + self.res(x))   # residual + BN
        return out


class PolygonEncoderPolyMP(nn.Module):
    def __init__(self, in_dim=6, hidden_dim=128, emb_dim=128, dropout=0.1):
        super().__init__()
        self.conv1 = PolyMPConv(in_dim,     hidden_dim)
        self.conv2 = PolyMPConv(hidden_dim, hidden_dim)
        self.conv3 = PolyMPConv(hidden_dim, hidden_dim)
        self.conv4 = PolyMPConv(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        # readout: concat mean + max -> 2*hidden_dim
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, emb_dim),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.conv1(x, edge_index)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = self.dropout(x)
        x = self.conv3(x, edge_index)
        x = self.dropout(x)
        x = self.conv4(x, edge_index)
        # dual readout
        g_mean = global_mean_pool(x, batch)
        g_max  = global_max_pool(x, batch)
        g = torch.cat([g_mean, g_max], dim=1)  # [B, 2*hidden]
        x = self.proj(g)
        x = F.normalize(x, dim=1)
        return x


model = PolygonEncoderPolyMP(
    in_dim=6,
    hidden_dim=128,
    emb_dim=128,
    dropout=0.1
).to(DEVICE)

print(model)
print("Device:", DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params:,}")

PolygonEncoderPolyMP(
  (conv1): PolyMPConv(
    (mlp_msg): Sequential(
      (0): Linear(in_features=12, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
    (mlp_update): Sequential(
      (0): Linear(in_features=134, out_features=128, bias=True)
      (1): ReLU()
    )
    (bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (res): Linear(in_features=6, out_features=128, bias=False)
  )
  (conv2): PolyMPConv(
    (mlp_msg): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
    (mlp_update): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): ReLU()
    )
    (bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (res): Identity()
  )
  (conv3): PolyMPConv(
    (mlp_msg): Sequential(
      (0): Lin

In [38]:
# Self-supervised pretraining: predict descriptor from graph
# Forces GNN to learn the same representation as the descriptor

pretrain_model = PolygonEncoderPolyMP(in_dim=6, hidden_dim=128, emb_dim=512, dropout=0.1).to(DEVICE)
pretrain_opt = torch.optim.Adam(pretrain_model.parameters(), lr=1e-3)

# targets: the 512-d descriptors for DB polygons
desc_targets = torch.tensor(X_db, dtype=torch.float32)  # [39970, 512]

db_graph_list = [graphs[i] for i in db_graph_rows]
loader_pre = DataLoader(db_graph_list, batch_size=256, shuffle=False)  # must be False

# inside the loop, replace the target lookup:
for batch_idx, batch in enumerate(loader_pre):
    batch = batch.to(DEVICE)
    pred  = pretrain_model(batch)
    start = batch_idx * 256
    end   = start + pred.size(0)
    target = F.normalize(desc_targets[start:end].to(DEVICE), dim=1)
    loss = 1 - (pred * target).sum(dim=1).mean()

for epoch in range(10):
    pretrain_model.train()
    total = 0
    for i, batch in enumerate(loader_pre):
        batch = batch.to(DEVICE)
        pred = pretrain_model(batch)  # [B, 512]
        start = i * 256
        target = desc_targets[start:start+len(pred)].to(DEVICE)
        loss = 1 - (pred * F.normalize(target, dim=1)).sum(dim=1).mean()
        pretrain_opt.zero_grad()
        loss.backward()
        pretrain_opt.step()
        total += loss.item()
    print(f"Pretrain epoch {epoch+1}/10 | loss={total/len(loader_pre):.4f}")

pretrain_sd = pretrain_model.state_dict()
model_sd = model.state_dict()

# copy everything except the final projection layer (size mismatch)
filtered = {k: v for k, v in pretrain_sd.items()
            if k in model_sd and v.shape == model_sd[k].shape}

model_sd.update(filtered)
model.load_state_dict(model_sd)
print(f"Pretrain weights transferred: {len(filtered)}/{len(model_sd)} tensors")

Pretrain epoch 1/10 | loss=0.2347
Pretrain epoch 2/10 | loss=0.1969
Pretrain epoch 3/10 | loss=0.1895
Pretrain epoch 4/10 | loss=0.1854
Pretrain epoch 5/10 | loss=0.1841
Pretrain epoch 6/10 | loss=0.1829
Pretrain epoch 7/10 | loss=0.1824
Pretrain epoch 8/10 | loss=0.1822
Pretrain epoch 9/10 | loss=0.1817
Pretrain epoch 10/10 | loss=0.1812
Pretrain weights transferred: 47/49 tensors


In [39]:
# ============================================================
# CELL 6 [FIX 4 + 5]: Train with semi-hard negatives,
# more positives per query, and longer training.
#
# FIX 4 — Semi-hard negatives:
#   v1 sampled negatives from all DB excluding only the top-100
#   GT. This gives trivially easy negatives (very dissimilar
#   polygons) so the triplet margin is satisfied instantly and
#   gradients vanish. Semi-hard negatives come from the
#   mid-ranked GT range (ranks 200-600) — similar enough to be
#   challenging but clearly non-relevant.
#
# FIX 5 — More positives + longer training:
#   v1 used only top-20 positives and 6 epochs (loss still
#   decreasing). We now use top-100 positives and 25 epochs
#   with cosine LR annealing.
# ============================================================

import random
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Batch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ----------------------------------------------------------
# Build positive pool from IoU-based GT
# Use top-100 positives (v1 used top-20)
# ----------------------------------------------------------
valid_graph_gid_set = set(graph_global_ids)
db_gid_set = set(db_graph_global_ids)

train_queries = []
positive_pool = {}

for q_gid, nn_list in custom_gt.items():
    if q_gid not in global_to_graphrow:
        continue
    # FIX: use top-100 positives instead of top-20
    pos = [gid for gid in nn_list[:100] if gid in db_gid_set and gid in global_to_graphrow]
    if len(pos) == 0:
        continue
    positive_pool[q_gid] = pos
    train_queries.append(q_gid)

print("Train queries with positives:", len(train_queries))

# ----------------------------------------------------------
# Semi-hard negative sampler
#
# Samples from GT ranks [semi_hard_min, semi_hard_max):
# these polygons are moderately similar to the query
# (not trivially far away) but not true positives.
# Falls back to rank > 500 if the semi-hard pool is small.
# ----------------------------------------------------------
SEMI_HARD_MIN = 200   # start of semi-hard zone in GT rank
SEMI_HARD_MAX = 600   # end of semi-hard zone in GT rank

db_candidates_all = np.array(db_graph_global_ids, dtype=np.int32)

def sample_negative(q_gid):
    nn_list = custom_gt.get(q_gid, [])
    # primary: semi-hard pool from GT ranks 200-600
    semi_hard_pool = [
        gid for gid in nn_list[SEMI_HARD_MIN:SEMI_HARD_MAX]
        if gid in global_to_graphrow
    ]
    if len(semi_hard_pool) >= 5:
        return random.choice(semi_hard_pool)
    # fallback: random beyond GT top-500
    banned = set(nn_list[:500]) | {q_gid}
    for _ in range(200):
        neg_gid = int(np.random.choice(db_candidates_all))
        if neg_gid not in banned and neg_gid in global_to_graphrow:
            return neg_gid
    # last resort
    return int(np.random.choice(db_candidates_all))


# ----------------------------------------------------------
# Triplet dataset
# ----------------------------------------------------------
class PolyTripletDataset(Dataset):
    def __init__(self, query_gids, positive_pool, repeats_per_query=5):
        self.samples = []
        for q_gid in query_gids:
            pos_list = positive_pool[q_gid]
            for _ in range(repeats_per_query):
                p_gid = random.choice(pos_list)
                n_gid = sample_negative(q_gid)
                self.samples.append((q_gid, p_gid, n_gid))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        q_gid, p_gid, n_gid = self.samples[idx]
        a = graphs[global_to_graphrow[q_gid]]
        p = graphs[global_to_graphrow[p_gid]]
        n = graphs[global_to_graphrow[n_gid]]
        return a, p, n


def collate_triplets(batch):
    return ([x[0] for x in batch],
            [x[1] for x in batch],
            [x[2] for x in batch])


# ----------------------------------------------------------
# Model + optimizer
# ----------------------------------------------------------
model = PolygonEncoderPolyMP(in_dim=6, hidden_dim=128, emb_dim=128, dropout=0.1).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

EPOCHS             = 25    # v1 used 6 (loss still decreasing)
BATCH_SIZE         = 64
REPEATS_PER_QUERY  = 5
MARGIN             = 0.3   # slightly larger margin for harder negatives

criterion = torch.nn.TripletMarginLoss(margin=MARGIN, p=2)

# cosine LR schedule: warm up then decay to 0
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-5
)

best_state = None
best_loss  = float("inf")

for epoch in range(1, EPOCHS + 1):
    # rebuild triplets each epoch so negatives refresh
    train_ds = PolyTripletDataset(train_queries, positive_pool,
                                  repeats_per_query=REPEATS_PER_QUERY)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_triplets
    )

    model.train()
    running_loss = 0.0
    num_batches  = 0

    for anchors, positives, negatives in train_loader:
        optimizer.zero_grad()
        a_batch = Batch.from_data_list(anchors).to(DEVICE)
        p_batch = Batch.from_data_list(positives).to(DEVICE)
        n_batch = Batch.from_data_list(negatives).to(DEVICE)

        z_a = model(a_batch)
        z_p = model(p_batch)
        z_n = model(n_batch)

        # embeddings are already L2-normalised inside the model,
        # but re-normalise defensively in case of fp noise
        z_a = F.normalize(z_a, p=2, dim=1)
        z_p = F.normalize(z_p, p=2, dim=1)
        z_n = F.normalize(z_n, p=2, dim=1)

        loss = criterion(z_a, z_p, z_n)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        num_batches  += 1

    scheduler.step()
    epoch_loss = running_loss / max(num_batches, 1)
    lr_now = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:2d}/{EPOCHS} | loss={epoch_loss:.4f} | lr={lr_now:.2e}")

    if epoch_loss < best_loss:
        best_loss  = epoch_loss
        best_state = copy.deepcopy(model.state_dict())

if best_state is not None:
    model.load_state_dict(best_state)

print("\nBest training loss:", best_loss)

Train queries with positives: 1000
Epoch  1/25 | loss=0.2746 | lr=9.96e-04
Epoch  2/25 | loss=0.2644 | lr=9.84e-04
Epoch  3/25 | loss=0.2582 | lr=9.65e-04
Epoch  4/25 | loss=0.2565 | lr=9.39e-04
Epoch  5/25 | loss=0.2540 | lr=9.05e-04
Epoch  6/25 | loss=0.2532 | lr=8.66e-04
Epoch  7/25 | loss=0.2448 | lr=8.21e-04
Epoch  8/25 | loss=0.2405 | lr=7.70e-04
Epoch  9/25 | loss=0.2356 | lr=7.16e-04
Epoch 10/25 | loss=0.2311 | lr=6.58e-04
Epoch 11/25 | loss=0.2267 | lr=5.98e-04
Epoch 12/25 | loss=0.2248 | lr=5.36e-04
Epoch 13/25 | loss=0.2160 | lr=4.74e-04
Epoch 14/25 | loss=0.2039 | lr=4.12e-04
Epoch 15/25 | loss=0.1931 | lr=3.52e-04
Epoch 16/25 | loss=0.1916 | lr=2.94e-04
Epoch 17/25 | loss=0.1265 | lr=2.40e-04
Epoch 18/25 | loss=0.1338 | lr=1.89e-04
Epoch 19/25 | loss=0.1407 | lr=1.44e-04
Epoch 20/25 | loss=0.1212 | lr=1.05e-04
Epoch 21/25 | loss=0.0935 | lr=7.12e-05
Epoch 22/25 | loss=0.1104 | lr=4.48e-05
Epoch 23/25 | loss=0.0950 | lr=2.56e-05
Epoch 24/25 | loss=0.1071 | lr=1.39e-05
Epoch

In [40]:
# ============================================================
# CELL 7: Generate embeddings from trained PolyMP
# (unchanged from v1)
# ============================================================

import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model.eval()

loader = DataLoader(graphs, batch_size=512, shuffle=False)

emb_list = []

with torch.no_grad():
    for batch in loader:
        batch = batch.to(DEVICE)
        h = model(batch)
        h = F.normalize(h, p=2, dim=1)
        emb_list.append(h.cpu())

H = torch.cat(emb_list, dim=0)

print("Embedding shape:", H.shape)

Embedding shape: torch.Size([49963, 128])


In [ ]:
# ============================================================
# CELL 8: PolyMP Recall@10/50/100/500 vs IoU-based GT
# (unchanged from v1 except now evaluated against proper GT)
# ============================================================

import numpy as np

H_np = H.numpy()
H_db = H_np[db_graph_rows]

def get_topk_cosine_db_only(query_gid, k=10):
    if query_gid not in global_to_graphrow:
        return []
    q_idx = global_to_graphrow[query_gid]
    q_vec = H_np[q_idx]
    sims = H_db @ q_vec
    topk_local = np.argpartition(-sims, kth=k-1)[:k]
    topk_local = topk_local[np.argsort(-sims[topk_local])]
    return [db_graph_global_ids[i] for i in topk_local]

def recall_poly_multi(k_values=[10, 50, 100, 500], max_queries=200):
    queries = list(custom_gt.keys())
    if max_queries is not None:
        queries = queries[:max_queries]

    results      = {}
    valid_counts = {}

    for k in k_values:
        scores = []
        for q in queries:
            if q not in custom_gt or len(custom_gt[q]) < k:
                continue
            pred = get_topk_cosine_db_only(q, k=k)
            if len(pred) < k:
                continue
            pred_set = set(pred)
            gt_set   = set(custom_gt[q][:k])
            scores.append(len(pred_set & gt_set) / k)
        results[k]      = float(np.mean(scores)) if scores else 0.0
        valid_counts[k] = len(scores)

    return results, valid_counts

K_VALUES = [10, 50, 100, 500]

recalls_poly, valid_counts_poly = recall_poly_multi(
    k_values=K_VALUES,
    max_queries=200
)

print("\n=== POLYMP RECALL (shape-similarity GT) ===")
for k in K_VALUES:
    print(f"PolyMP Recall@{k:3d}: {recalls_poly[k]:.4f} | valid_queries={valid_counts_poly[k]}")

# --- baseline comparison: descriptor-cosine recall (upper bound for hand-crafted) ---
print("\n=== DESCRIPTOR BASELINE (cosine on hand-crafted 128-d) ===")
H_desc_db = X_db   # already L2-normalised
H_desc_q  = X_q

for k in K_VALUES:
    scores = []
    for r, q_gid in enumerate(list(custom_gt.keys())[:200]):
        if len(custom_gt[q_gid]) < k:
            continue
        sims_d = H_desc_q[r] @ H_desc_db.T
        topk   = np.argpartition(-sims_d, kth=k-1)[:k]
        topk   = topk[np.argsort(-sims_d[topk])]
        pred_set = set(int(db_global_ids[i]) for i in topk)
        gt_set   = set(custom_gt[q_gid][:k])
        scores.append(len(pred_set & gt_set) / k)
    print(f"Descriptor Recall@{k:3d}: {np.mean(scores):.4f}")


=== POLYMP v2 RESULT (IoU GT) ===
PolyMP Recall@ 10: 0.0415 | valid_queries=200
PolyMP Recall@ 50: 0.0545 | valid_queries=200
PolyMP Recall@100: 0.0596 | valid_queries=200
PolyMP Recall@500: 0.0755 | valid_queries=200

=== DESCRIPTOR BASELINE (cosine on hand-crafted 128-d) ===
Descriptor Recall@ 10: 1.0000
Descriptor Recall@ 50: 0.9999
Descriptor Recall@100: 1.0000
Descriptor Recall@500: 1.0000


In [42]:
# sanity check: rotate a polygon 90deg, descriptor should be ~identical
import numpy as np
from shapely.affinity import rotate
geom = geometries_valid[0]
geom_rot = rotate(geom, 90, origin='centroid')
d1 = _descriptor_from_geom(geom)
d2 = _descriptor_from_geom(geom_rot)
print("Cosine similarity after 90deg rotation:", float(d1 @ d2))
# should be close to 1.0 — if it's not, PCA flip ambiguity is the culprit

Cosine similarity after 90deg rotation: 0.7977970242500305


In [43]:
# quick spot check
test_q = train_queries[0]
negs = [sample_negative(test_q) for _ in range(20)]
nn_list = custom_gt[test_q]
ranks = [nn_list.index(n) if n in nn_list else 9999 for n in negs]
print("Negative GT ranks:", ranks)
# should see values in 200-600 range, not all 9999

Negative GT ranks: [218, 256, 344, 277, 211, 263, 224, 410, 474, 249, 370, 249, 236, 282, 457, 461, 328, 436, 313, 236]
